In [1]:
print("Hello freaky Nikki")

Hello freaky Nikki


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from dstapi import DstApi

# autoreload modules when code is run
%load_ext autoreload
%autoreload 2

# user written modules
import dataproject

ModuleNotFoundError: No module named 'dataproject'

## Question 1

# 1.1 The Gini coefficient and the top 10 percent share

In [ ]:
# a. connect to the two tables
ifor41 = DstApi('IFOR41')   # inequality measures, incl. the Gini coefficient
ifor32 = DstApi('IFOR32')   # average income by decile group

# b. inspect what the tables contain
for name, tab in [('IFOR41', ifor41), ('IFOR32', ifor32)]:
    print('=' * 60)
    print(name)
    print('=' * 60)
    print(tab.tablesummary(language='en'))

In [ ]:
gini = (gini_raw
        .rename(columns={'TID':'year', 'INDHOLD':'gini'})
        .loc[:, ['year','gini']]
        .astype({'year':int, 'gini':float})
        .sort_values('year')
        .reset_index(drop=True))

# b. clean IFOR32: one row per year and decile group
dec = (dec_raw
       .rename(columns={'TID':'year', 'INDHOLD':'mean_income', 'DECILGEN':'decile'})
       .loc[:, ['year','decile','mean_income']]
       .astype({'year':int, 'mean_income':float}))

# c. the top 10 pct. share: mean income in the 10th decile relative to the sum of all deciles
total = dec.groupby('year')['mean_income'].sum()
top   = dec[dec['decile'].str.contains('tenth', case=False)].set_index('year')['mean_income']
assert len(top) == len(total), 'the top decile group was not identified correctly'

top10 = (100*top/total).rename('top10').reset_index()

# d. merge, checking that no rows are duplicated
data = pd.merge(gini, top10, on='year', validate='1:1')

print(data.head(3))
print(data.tail(3))
print('correlation:', round(data['gini'].corr(data['top10']), 3))

In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))

ax.plot(data['year'], data['gini'],  label='Gini coefficient')
ax.plot(data['year'], data['top10'], label='Top 10 pct. income share')

ax.set_xlabel('year')
ax.set_ylabel('pct.')
ax.set_title('Inequality in Denmark, 1987-2024')
ax.legend(frameon=False)
ax.grid(alpha=0.3)

fig.tight_layout()

Inequality in Denmark has risen clearly since 1987. The Gini coefficient went from 22.1 to 30.4 pct., and the share of income going to the richest 10 pct. went from 18.5 to 25.1 pct. Both are around a third higher than at the start.

The increase is not steady. Both series are flat until the mid-1990s, rise slowly until 2003, and then rise much faster from 2004 onwards. Both drop in 2009 and recover afterwards. This is the financial crisis, where capital income fell sharply. Capital income is concentrated at the top of the distribution.

The two measures tell the same story. Their correlation over time is 0.98, and the turning points appear in both. This makes sense, because the top decile has a large weight in the Gini coefficient. They are still not the same thing: the Gini coefficient uses the whole distribution, while the top 10 pct. share only looks at the top. That they move together suggests the rise in inequality comes mainly from the top pulling away.

# 1.3 Municipalities

In [ ]:
# a. IFOR41 for all municipalities
p41 = ifor41.define_base_params(language='en')
for var in p41['variables']:
    code = var['code'].upper()
    if code == 'ULLIG':       var['values'] = ['70']
    elif code == 'KOMMUNEDK': var['values'] = ['*']
    elif code == 'TID':       var['values'] = ['*']

gini_kom_raw = ifor41.get_data(params=p41)

# b. IFOR32 for all municipalities
p32 = ifor32.define_base_params(language='en')
for var in p32['variables']:
    code = var['code'].upper()
    if code == 'DECILGEN':    var['values'] = ['*']
    elif code == 'KOMMUNEDK': var['values'] = ['*']
    elif code == 'TID':       var['values'] = ['*']

dec_kom_raw = ifor32.get_data(params=p32)

print('IFOR41:', gini_kom_raw.shape)
print('IFOR32:', dec_kom_raw.shape)
print(gini_kom_raw['KOMMUNEDK'].nunique(), 'municipality codes')

In [ ]:
gini_kom = (gini_kom_raw
            .rename(columns={'KOMMUNEDK':'mun', 'TID':'year', 'INDHOLD':'gini'})
            .loc[:, ['mun','year','gini']]
            .astype({'year':int, 'gini':float})
            .query("mun != 'All Denmark'"))

# b. clean IFOR32 and compute the top 10 pct. share per municipality and year
dec_kom = (dec_kom_raw
           .rename(columns={'KOMMUNEDK':'mun', 'TID':'year',
                            'INDHOLD':'mean_income', 'DECILGEN':'decile'})
           .loc[:, ['mun','year','decile','mean_income']]
           .astype({'year':int, 'mean_income':float})
           .query("mun != 'All Denmark'"))

total_kom = dec_kom.groupby(['mun','year'])['mean_income'].sum()
top_kom = (dec_kom[dec_kom['decile'].str.contains('tenth', case=False)]
           .set_index(['mun','year'])['mean_income'])

top10_kom = (100*top_kom/total_kom).rename('top10').reset_index()

# c. merge
kom = pd.merge(gini_kom, top10_kom, on=['mun','year'], validate='1:1')

print(kom.shape, '=', kom['mun'].nunique(), 'municipalities x', kom['year'].nunique(), 'years')
print(kom.head(3))
print('missing:', kom.isna().sum().sum())

In [ ]:
last_year = kom['year'].max()
cross = kom[kom['year'] == last_year]

print(f'year: {last_year}, N = {len(cross)}')
print('correlation across municipalities:', round(cross['gini'].corr(cross['top10']), 3))

fig, ax = plt.subplots(figsize=(6,5))
ax.scatter(cross['gini'], cross['top10'], s=25, alpha=0.7)
ax.set_xlabel('Gini coefficient (pct.)')
ax.set_ylabel('Top 10 pct. income share (pct.)')
ax.set_title(f'Municipalities, {last_year}')
ax.grid(alpha=0.3)
fig.tight_layout()